In [3]:
import numpy as np
import pandas as pd
import scipy.stats as stats

## Time Series Classification: Window Splitting
Window size and step size can be changed below. To keep as much data as possible, we have selected a step size of 1. To create window of ~25 seconds, we've selected a window size of 5.

In [10]:
#Window size, approx. 7 second
window_size = 35

#Step size
step_size = 1

#Participant ID 
pid = "011" #2, 5, 6, 9, 10, 11, 13, 15, 16, 17, 18, 19, 20, 24

# Task condition
condition = "RW"

In [11]:
participant_df = pd.read_csv("/Users/juliasantaniello/Desktop/fNIRS-2-RL/Experiment/ParticipantData/fNIRS/LabeledData/{}_{}_LabeledData.csv".format(pid, condition))

labels = ['continuous_optimal', 'binary_optimal', 'discrete_optimal']

#Features being used
all_neural_features = ['L_O_DSphi', 'L_D_DSphi', 'R_D_DSphi', 'R_O_DSphi', 'L_O_DSI', 'R_D_DSI', 'R_O_DSI', 'L_D_DSI']
phasic_neural_features = ['L_O_DSphi', 'L_D_DSphi', 'R_D_DSphi', 'R_O_DSphi']
intensity_neural_features = ['L_O_DSI', 'R_D_DSI', 'R_O_DSI', 'L_D_DSI']

In [12]:

feature_map = {0:"Mean",
                1:"Std",
                2:"Slope",
                3:"Intercept",
                4:"Kurtosis",
                5:"Skewness"
                }

def create_window_df(df, window_size:int, step_size:int, features, labels):
    data = {}

    # Add features
    for feature in features:
        data[feature] = df[feature]
    
    # Add labels
    for label in labels:
        data[label] = df[label]

    df = pd.DataFrame(data)

    windowed_data = []
    windowed_labels = []
    slope_values = []
    intercept_values = []
    start_timestamps = []
    end_timestamps = []

    for start in range(0, len(df) - window_size + 1, step_size):
        data_dict = {}
        end = start + window_size
        window = df.iloc[start:end] 

        last_discrete_label = window["discrete_optimal"].iloc[-1]
        last_continuous_label = window['continuous_optimal'].iloc[-1]
        last_binary_label = window['binary_optimal'].iloc[-1]

        window = window.drop(columns=['continuous_optimal', 'binary_optimal', 'discrete_optimal'])

        # Calculate window features
        mean_values = window.mean(axis=0).to_numpy(dtype=float)
        std_values = window.std(axis=0).to_numpy(dtype=float)
        slope_values = np.array([np.polyfit(window[feature], np.arange(window_size), 1)[0] for feature in features], dtype=float)
        intercept_values = np.array([np.polyfit(window[feature], np.arange(window_size), 1)[1] for feature in features], dtype=float)
        kurtosis_values = stats.kurtosis(window, axis=0, fisher=True)
        skewdness_values = stats.skew(window, axis=0)

        # Add features
        for i, feature in enumerate(features):
            for j, stat in feature_map.items():
                data_dict[f"{feature}_{stat}"] = np.array([mean_values[i], std_values[i], slope_values[i], intercept_values[i], kurtosis_values[i], skewdness_values[i]])[j]

        # Add start and end timestamps
        start_timestamps.append(participant_df['time'].iloc[start])
        end_timestamps.append(participant_df['time'].iloc[end - 1])

        windowed_data.append(data_dict)
        windowed_labels.append({'discrete_label': last_discrete_label, 'continuous_label': last_continuous_label, 'binary_label': last_binary_label})

    windowed_data = pd.DataFrame(windowed_data)
    windowed_labels_df = pd.DataFrame(windowed_labels)

    # Add start and end timestamps to the windowed data
    windowed_data['start_timestamp'] = start_timestamps
    windowed_data['end_timestamp'] = end_timestamps

    return windowed_data, windowed_labels_df

In [ ]:
window_data, window_labels = create_window_df(participant_df, window_size=window_size, step_size=1, labels=labels, features=all_neural_features)

#concatenate labels and windowed data
df_concat_rows = pd.concat([window_labels, window_data], axis=1)

df_concat_rows["pid"] = pid
df_concat_rows["condition"] = condition


In [ ]:
#concatenate labels and windowed data

df_concat_rows.tail(2)

,discrete_label,continuous_label,binary_label,L_O_DSphi_Mean,L_O_DSphi_Std,L_O_DSphi_Slope,L_O_DSphi_Intercept,L_O_DSphi_Kurtosis,L_O_DSphi_Skewness,L_D_DSphi_Mean,...,L_D_DSI_Mean,L_D_DSI_Std,L_D_DSI_Slope,L_D_DSI_Intercept,L_D_DSI_Kurtosis,L_D_DSI_Skewness,start_timestamp,end_timestamp,pid,condition
977,0,0.129708,0,8.632616,0.747316,11.540719,-82.626595,-1.488966,-0.477634,-1.602474,...,-0.294173,0.150372,54.344637,32.986723,-0.579994,0.931074,2024-09-19 21:16:06.163310080,2024-09-19 21:16:15.709319168,011,RW
978,0,0.129708,0,8.663483,0.711262,11.056519,-78.787956,-1.480009,-0.482793,-1.599148,...,-0.284954,0.160214,54.331888,32.482103,-0.815215,0.848178,2024-09-19 21:16:06.240521984,2024-09-19 21:16:15.762342912,011,RW


In [ ]:
# Save as csv file
df_concat_rows.to_csv("{}_{}_{}Windows".format(pid, condition, window_size), index=True)